# Section 3: Inflation and localization

*(Replaces DART_LAB slide deck Section 3.)*

Two systematic problems degrade ensemble filters in real systems:

1. **Too little spread** — model error and sampling make the ensemble
   overconfident; the filter then ignores observations. Fix: **variance
   inflation**.
2. **Spurious correlations** — with a finite ensemble, distant, unrelated
   variables show nonzero *sample* correlation and get spurious updates.
   Fix: **localization**.

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import pydartlab as dl
import pydartlab.apps as apps

## Diagnosing spread: rank histograms

Sort the $N$ ensemble members; they divide the line into $N{+}1$ bins. If
the ensemble is statistically consistent, the truth is equally likely to
fall in any bin, so accumulating the truth's *rank* over many cycles gives
a **flat** histogram.

* **U-shape** — truth keeps falling outside: ensemble has *too little
  spread* (the common case).
* **Dome** — too much spread.
* **Ramp** — bias.

## Exercise: rank histograms in `oned_model`

Auto-run a few hundred steps with defaults and check the rank histograms
are roughly flat. Then set **Bias = 1** and watch the U-shape (actually a
ramp toward one side) develop.

In [ ]:
om = apps.oned_model()
om

## Variance inflation

Multiply ensemble deviations from the mean by $\sqrt{\lambda}$, $\lambda > 1$:

$$ x_n \leftarrow \sqrt{\lambda}\,(x_n - \bar{x}) + \bar{x} $$

The mean is unchanged; the variance becomes $\lambda \sigma^2$. This gives
observations more weight and restores consistency when the model is wrong.

## Exercise: inflation in `oned_ensemble`

Create an ensemble, enable inflation, set a value > 1, and **Update**: the
orange (inflated) ensemble and its posterior appear below the original.
Compare posterior means/SDs with and without inflation, especially when
the prior is far from the observation.

In [ ]:
oe = apps.oned_ensemble()
oe.set_ensemble([2.0, 2.4, 2.8, 3.2])  # small spread, far from obs at 1.0
oe.inf_toggle.value = True
oe.inf_slider.value = 2.5
oe.update_ensemble()
oe

## Exercise: inflation against model bias in `oned_model`

1. Set **Bias = 1**, inflation = 1 (none); run. Error >> spread, U-shaped
   histogram.
2. Set **Inflation = 1.5**; run again. Error and spread come together and
   the histogram flattens.
3. What happens with far too much inflation (e.g. 4)?
4. Try the same with the nonlinear parameter *a* > 0 instead of bias.

In [ ]:
om2 = apps.oned_model()
om2

## Regression sampling error and localization

The regression coefficient $\widehat{\mathrm{cov}}(x,y)/\widehat{\mathrm{var}}(y)$ is
estimated from $N$ samples. If the *true* correlation is 0, the *sample*
correlation is not — it has magnitude $\sim 1/\sqrt{N}$. Run the cell
below to see it.

In [ ]:
# Monte-Carlo: sample correlation of two UNCORRELATED variables
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(6.5, 3))
sizes = [5, 10, 20, 40, 80, 160]
mean_abs = [np.mean([abs(np.corrcoef(rng.normal(size=n), rng.normal(size=n))[0, 1])
                     for _ in range(2000)]) for n in sizes]
ax.plot(sizes, mean_abs, "o-")
ax.plot(sizes, 0.8 / np.sqrt(sizes), "--", label=r"$\propto 1/\sqrt{N}$")
ax.set_xlabel("ensemble size N"); ax.set_ylabel("mean |sample correlation|")
ax.legend(); ax.set_title("Spurious correlation from finite ensembles");

In a 40-variable model, every observation updates **every** variable
through these noisy regressions; for distant variables the true correlation
is ~0 and the update is pure noise. **Localization** tapers the regression
with distance using the Gaspari-Cohn function — a smooth, compactly
supported polynomial that is 1 at distance 0 and exactly 0 beyond twice the
half-width $c$:

In [ ]:
dist = np.linspace(0, 0.6, 200)
fig, ax = plt.subplots(figsize=(6.5, 3))
for c in (0.1, 0.2, 0.3):
    ax.plot(dist, dl.comp_cov_factor(dist, c), label=f"half-width c = {c}")
ax.set_xlabel("distance (fraction of domain)"); ax.set_ylabel("regression factor")
ax.legend(); ax.set_title("Gaspari-Cohn localization");

## Exercise: localization, ensemble size and inflation in `run_lorenz_96`

The localization control is the Gaspari-Cohn half-width as a *fraction of
the domain*.

1. EAKF with **localization 0.2**: confirm in the state panel that an
   observation no longer perturbs the far side of the cyclic domain.
2. Turn localization off (set it huge, e.g. 1000000) and run with the
   default 20 members. Then raise the ensemble to 80: a big ensemble can
   live without localization.
3. Drop to **10 members** and find the localization that works best.
4. Add inflation (1.1-1.5) on top of localization.
5. Set **Forcing = 6** for the ensemble (truth stays at 8 — model error!)
   and see how much inflation it takes to compensate.

In [ ]:
l96 = apps.run_lorenz_96(seed=4)
l96

In [ ]:
# Scripted sweep: error vs localization for two ensemble sizes
from pydartlab.experiments import Lorenz96Experiment

halfwidths = [0.05, 0.1, 0.2, 0.4, 1e6]
fig, ax = plt.subplots(figsize=(7, 3.2))
for n in (10, 40):
    errs = []
    for c in halfwidths:
        exp = Lorenz96Experiment(filter_type="EAKF", ens_size=n,
                                 localization=c, seed=7)
        for _ in range(120):
            exp.step()
        errs.append(np.mean(exp.history["post_error"][-40:]))
    ax.plot(range(len(halfwidths)), errs, "o-", label=f"N = {n}")
ax.set_xticks(range(len(halfwidths)),
              [str(c) if c < 1 else "none" for c in halfwidths])
ax.set_xlabel("localization half-width"); ax.set_ylabel("posterior error")
ax.legend(); ax.set_title("Small ensembles need localization");

## What you should have seen

* U-shaped rank histograms diagnose insufficient spread; inflation fixes
  it (and too much inflation over-disperses).
* Sample correlations of uncorrelated variables scale as $1/\sqrt{N}$ —
  finite ensembles *will* produce spurious updates.
* Localization makes small ensembles viable; the best half-width depends
  on ensemble size and model error.

**Next: Section 4 — what if the variable can't be negative, or the prior
isn't Gaussian at all?**